# BirdCLEF 2026 — Baseline Model Experiments

In [ ]:
import sys
sys.path.append('..')

import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold

from src.config import load_config, ROOT
from src.dataset import BirdCLEFDataset
from src.models import BirdCLEFModel
from src.train import run_training
from src.utils import seed_everything

In [ ]:
config = load_config('../configs/baseline.yaml')
seed_everything(config['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
df = pd.read_csv(ROOT / config['train_csv'])
species_list = sorted(df['primary_label'].unique().tolist())
print(f'Species: {len(species_list)}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=config['seed'])
splits = list(skf.split(df, df['primary_label']))
train_idx, val_idx = splits[config['val_fold']]

audio_dir = ROOT / config['train_audio_dir']
train_ds = BirdCLEFDataset(df.iloc[train_idx], audio_dir, species_list, config, train=True)
val_ds   = BirdCLEFDataset(df.iloc[val_idx],   audio_dir, species_list, config, train=False)

train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,  num_workers=config['num_workers'])
val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])

In [ ]:
model = BirdCLEFModel(num_classes=len(species_list), model_name=config['model_name'], pretrained=config['pretrained'])
model = model.to(device)
print(model)

In [ ]:
output_dir = ROOT / config['output_dir']
run_training(model, train_loader, val_loader, config, output_dir, device)